# Estudo Comparativo de Classificação Acústica Submarina — Dataset IARA
### Notebook 18: Análise de Ortogonalidade, Efeito Oráculo e Grau de Consenso no Random Forest — Classe MEDIUM

**Autor:** Evandro Rocha (UFRJ / COPPE / PEE)
**Data:** Outubro de 2026

---

## Resumo Executivo da Investigação com Random Forest (RF)

A classe **`MEDIUM`** (rebocadores e navios de apoio offshore de $50\\text{ m}$ a $100\\text{ m}$) foi avaliada em todas as **1.490 instâncias de teste** nos 10 folds oficiais do protocolo $5\\times2$cv com a arquitetura **Random Forest (100 árvores com balanceamento de classes)** nas 4 representações (**MEL, LOFAR, HHT e DEMON**).

O objetivo é confrontar os resultados do classificador SVM (Notebook 12) e do MLP (Notebook 15) com os do Random Forest para avaliar a estabilidade do consenso em navios de porte intermediário e o padrão de confusão com lanchas rápidas.

### Principais Descobertas no Random Forest:
1. **Teto Teórico do Oráculo RF Atinge $77{,}85\\%$ (1.160 / 1.490 rebocadores):**
   - O Oráculo do Random Forest em `MEDIUM` proporciona um salto de **$+19{,}1$ pontos percentuais** sobre o melhor modelo isolado (MEL com $58{,}72\\%$).
   - O erro comum irrecuperável é de **$22{,}15\\%$ (330 áudios)**.
2. **A Unanimidade em MEDIUM se Mantém Estável ($26{,}78\\%$ — 399 áudios):**
   - Assim como no SVM ($27{,}05\\%$) e no MLP ($29{,}93\\%$), cerca de mais de um quarto dos rebocadores são classificados corretamente pelas 4 representações de forma independente da transformada.
3. **MEL e HHT Lideram os Resgates Exclusivos no RF:**
   - O **MEL salva sozinho 99 rebocadores ($6{,}64\\%$)** e o **HHT salva 93 ($6{,}24\\%$)**, demonstrando que transformadas espectrais diretas e tempo-frequência contínua superam o DEMON na caracterização de propulsão de apoio offshore.
4. **A Confusão Extrema `MEDIUM` $\\rightarrow$ `SMALL` no Random Forest:**
   - No Random Forest, quando o modelo erra em `MEDIUM`, **mais de $65\\%$ a $72\\%$ dos erros vão para `SMALL`** (ex: no MEL, $72{,}03\\%$ dos erros vão para `SMALL`). As árvores de decisão particionam a rotação de motores diesel de média rotação em nós muito próximos aos de propulsores de lanchas pesadas.

In [ ]:
# 1. Configuração do Ambiente e Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

# Estilização publication-ready (sem dependência externa)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 13,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 15,
    'figure.dpi': 150
})

print("Ambiente configurado com sucesso! Bibliotecas prontas.")

## 2. Metodologia Experimental e Base de Dados de Teste

Avaliação sobre todos os **1.490 testes** onde a classe real comprovada era **`MEDIUM` (Classe 1)** sob o protocolo oficial $5\\times2$cv nos 10 folds com modelos **Random Forest (100 árvores, balanced)**:


In [ ]:
# 2. Dados Consolidados das Predições do Random Forest na Classe MEDIUM
total_med = 1490

# Desempenho individual no RF
hits_data = {
    'Representacao': ['DEMON (253-D)', 'LOFAR (1024-D)', 'HHT Puro (256-D)', 'MEL (256-D)', 'ORÁCULO RF (União 4 Modos)'],
    'Acertos': [520, 821, 840, 875, 1160],
    'Recall_Pct': [34.90, 55.10, 56.38, 58.72, 77.85]
}
df_hits = pd.DataFrame(hits_data)

# Grau de consenso no RF
consensus_data = {
    'Nivel_Consenso': [
        '4 de 4 (Unanimidade / Independente)',
        '3 de 4 (Alto Consenso)',
        '2 de 4 (Consenso Parcial / Dividido)',
        '1 de 4 (Frágil / Resgate Exclusivo)',
        '0 de 4 (Erro Comum / Incorrigível)'
    ],
    'Audios': [399, 239, 221, 301, 330],
    'Pct_Total_Medium': [26.78, 16.04, 14.83, 20.20, 22.15],
    'Pct_Dentro_Oraculo': [34.40, 20.60, 19.05, 25.95, np.nan]
}
df_consensus = pd.DataFrame(consensus_data)

# Resgates exclusivos por representação no RF
exclusive_data = {
    'Representacao': ['DEMON Exclusivo', 'LOFAR Exclusivo', 'HHT Exclusivo', 'MEL Exclusivo'],
    'Audios_Salvos': [34, 75, 93, 99],
    'Percentual': [2.28, 5.03, 6.24, 6.64]
}
df_exclusive = pd.DataFrame(exclusive_data)

# Efeito Reverso em Relação ao Baseline MEL (875 acertos, 615 erros)
reverse_data = {
    'Destino_Troca': ['Para LOFAR', 'Para HHT', 'Para DEMON'],
    'Mantem_do_MEL': [671, 677, 446],
    'Perde_do_MEL (Efeito Reverso)': [204, 198, 429],
    'Pct_Perda_Acertos (%)': [23.31, 22.63, 49.03],
    'Resgata_Erros_MEL': [150, 163, 74],
    'Pct_Resgate_Erros (%)': [24.39, 26.50, 12.03],
    'Saldo_Liquido_Audios': [-54, -35, -355]
}
df_reverse = pd.DataFrame(reverse_data)

# Padrão de confusão quando erra em MEDIUM no RF
confusion_pattern_data = {
    'Representacao': ['MEL (615 erros)', 'LOFAR (669 erros)', 'HHT (650 erros)', 'DEMON (970 erros)'],
    'Confundido_SMALL_Pct': [72.03, 65.77, 64.62, 67.32],
    'Confundido_LARGE_Pct': [27.64, 33.78, 31.54, 27.42],
    'Confundido_AMBIENTAL_Pct': [0.33, 0.45, 3.85, 5.26]
}
df_confusion_pattern = pd.DataFrame(confusion_pattern_data)

display(df_hits)
display(df_consensus)
display(df_reverse)

## 3. O Efeito Oráculo no Random Forest: Recall Individual vs. Teto Teórico

O classificador Random Forest atinge **$58{,}72\\%$ de recall em `MEDIUM`** com o MEL e **$56{,}38\\%$** com o HHT.

Contudo, a combinação das 4 entradas via **Oráculo RF** salta para **$77{,}85\\%$ (1.160 acertos)**:
- Um ganho de **$+19{,}1$ pontos percentuais** sobre o MEL isolado;
- O erro comum (onde as 4 florestas erram simultaneamente) é de **$22{,}15\\%$ (330 áudios)**.

In [ ]:
# 3. Gráfico: Desempenho Isolado vs Teto Teórico do Oráculo no RF em MEDIUM
plt.figure(figsize=(10, 5.5))
colors = ['#95a5a6', '#e67e22', '#9b59b6', '#2980b9', '#27ae60']
bars = plt.bar(df_hits['Representacao'], df_hits['Recall_Pct'], color=colors, width=0.55, edgecolor='black', linewidth=0.8)
bars[-1].set_hatch('//')

plt.axhline(77.85, color='#27ae60', linestyle='--', lw=1.5, alpha=0.7, label='Teto do Oráculo RF (77.85%)')
plt.axhline(58.72, color='#2980b9', linestyle=':', lw=1.5, alpha=0.7, label='Melhor Isolado MEL (58.72%)')

plt.title('Recall Random Forest na Classe MEDIUM: Modelos Isolados vs. Teto Teórico do Oráculo', fontweight='bold', pad=12)
plt.ylabel('Recall em MEDIUM (%)')
plt.ylim(0, 100)

for bar in bars:
    h = bar.get_height()
    plt.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width()/2, h),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom',
                 fontsize=10.5, fontweight='bold', color='#111')

plt.legend(loc='upper left', frameon=True)
plt.tight_layout()
plt.show()

## 4. Análise de Grau de Consenso em MEDIUM no Random Forest

A pirâmide de consenso em `MEDIUM`:
- **Unanimidade (4/4 certas): $399$ áudios ($26{,}78\\%$)** — taxa estável com as outras famílias;
- **Alto Consenso (3/4 certas): $239$ áudios ($16{,}04\\%$)**;
- **Consenso Dividido (2/4 certas): $221$ áudios ($14{,}83\\%$)**;
- **Resgate Exclusivo (1/4 certa): $301$ áudios ($20{,}20\\%$)**;
- **Erro Comum (0/4 certas): $330$ áudios ($22{,}15\\%$)**.

### Votação por Consenso em MEDIUM (RF):
- **Unanimidade (4/4):** $26{,}78\\%$ ($399$ áudios);
- **Maioria Estrita ($\\ge 3/4$):** $42{,}82\\%$ ($638$ áudios);
- **Maioria Simples / Empate ($\\ge 2/4$):** $57{,}65\\%$ ($859$ áudios);
- **Oráculo ($\\ge 1/4$):** **$77{,}85\\%$ ($1.160$ áudios)**.

In [ ]:
# 4. Visualização do Grau de Consenso no RF
plt.figure(figsize=(9, 5))
labels_cons = ['4 Entradas\n(Unânime)', '3 Entradas\n(Alto Consenso)', '2 Entradas\n(Dividido)', '1 Entrada\n(Exclusivo/Frágil)', '0 Entradas\n(Erro Comum)']
counts_cons = [399, 239, 221, 301, 330]
pcts_cons = [26.78, 16.04, 14.83, 20.20, 22.15]
colors_cons = ['#27ae60', '#2980b9', '#f39c12', '#e74c3c', '#7f8c8d']

bars = plt.bar(labels_cons, counts_cons, color=colors_cons, edgecolor='black', linewidth=0.8, width=0.55)
plt.title('Grau de Consenso RF em MEDIUM (Total: 1.490 Áudios no 5x2cv)', fontweight='bold', pad=12)
plt.ylabel('Quantidade de Áudios de MEDIUM')
plt.ylim(0, 480)

for bar, pct in zip(bars, pcts_cons):
    h = bar.get_height()
    plt.annotate(f'{h} ({pct:.1f}%)', xy=(bar.get_x() + bar.get_width()/2, h),
                 xytext=(0, 4), textcoords="offset points", ha='center', va='bottom',
                 fontsize=9.5, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Resgates Exclusivos e Padrão de Erro no Random Forest

### Resgates Exclusivos em MEDIUM:
- **MEL:** Salva $99$ rebocadores ($6{,}64\\%$) — campeão em MEDIUM;
- **HHT:** Salva $93$ rebocadores ($6{,}24\\%$);
- **LOFAR:** Salva $75$ rebocadores ($5{,}03\\%$);
- **DEMON:** Salva $34$ rebocadores ($2{,}28\\%$).

### Padrão de Erro Unidirecional Maciço para SMALL:
- **MEL:** **$72{,}03\\%$ dos erros vão para `SMALL`**;
- **DEMON:** **$67{,}32\\%$ dos erros vão para `SMALL`**;
- **LOFAR:** **$65{,}77\\%$ dos erros vão para `SMALL`**;
- **HHT:** **$64{,}62\\%$ dos erros vão para `SMALL`**.
O Random Forest amplifica o viés de confundir rebocadores com lanchas, mostrando que em árvores de decisão os níveis de ruído ambiente e de marulho induzem o algoritmo a atribuir assinaturas acústicas intermediárias à classe `SMALL`.

In [ ]:
# 5. Exibição do Gráfico Oficial Consolidado salvo em disco
from IPython.display import Image, display

img_path = 'reports/analise_complementaridade_rf_medium_oraculo.png'
if os.path.exists(img_path):
    display(Image(filename=img_path))
else:
    print(f"Imagem {img_path} não encontrada. Execute 'scratch/generate_rf_predictions_and_analysis.py' para gerá-la.")

## 6. Comparativo Trilateral em MEDIUM: SVM Nyström vs. MLP PyTorch vs. Random Forest

| Dimensão Analítica | SVM Nyström (Notebook 12) | MLP PyTorch (Notebook 15) | Random Forest (Notebook 18) | Diagnóstico Científico |
| :--- | :---: | :---: | :---: | :--- |
| **Melhor Modelo Isolado** | $60{,}40\\%$ (MEL / LOFAR) | **$64{,}09\\%$ (MEL)** | $58{,}72\\%$ (MEL) | O MLP continua sendo o melhor classificador contínuo em `MEDIUM`. |
| **Teto do Oráculo** | $83{,}69\\%$ ($1.247$) | **$84{,}90\\%$ ($1.265$)** | $77{,}85\\%$ ($1.160$) | O RF tem teto inferior em MEDIUM devido à forte atração gravitacional por `SMALL`. |
| **Taxa de Unanimidade (4/4)** | $27{,}05\\%$ ($403$) | **$29{,}93\\%$ ($446$)** | $26{,}78\\%$ ($399$) | **Convergência total:** unanimidade em torno de $27	ext{--}30\\%$ nas 3 famílias. |
| **Erro Comum Incorrigível** | $16{,}31\\%$ ($243$) | **$15{,}10\\%$ ($225$)** | $22{,}15\\%$ ($330$) | O RF apresenta maior perda intrínseca em rebocadores. |
| **Votação Simples ($\\ge 2/4$)** | $64{,}63\\%$ ($+4{,}2$ p.p.) | **$65{,}84\\%$ ($+1{,}75$ p.p.)** | $57{,}65\\%$ | SVM e MLP superam modelos isolados por votação simples em `MEDIUM`. |